# Original ApET: full GQA balanced Test-Dev

Review the pilot first. This is 12,578 questions on 398 images, four configurations, 50,312 predictions. Enable Internet and T4 ×2. See `research/GQA_FULL_TESTDEV.md`. Native code, locked dependencies, checkpoint revisions and pilot scoring remain unchanged. Default start flag blocks accidental execution on import.

## 1. Start after pilot review, or restore a previous run


In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

RUN_FULL_TEST = False  # Set True after reviewing the pilot results.
RESUME_RUN_DIR = None  # Optional extracted prior run: "/kaggle/working/restored-gqa"
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
if not RUN_FULL_TEST:
    raise RuntimeError("Full run is prepared for review. Set RUN_FULL_TEST=True after reviewing the GQA pilot.")

repo = Path("/kaggle/working/CSE590-ApET")
if RESUME_RUN_DIR:
    run_dir = Path(RESUME_RUN_DIR)
    saved = json.loads((run_dir / "original-gqa-full.json").read_text())
    REPO_REF = saved["commit"]  # Restore inference/scoring source, not the newest branch.
else:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run_dir = Path(f"/kaggle/working/original-gqa-full-{stamp}")
    run_dir.mkdir()
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True)
if not RESUME_RUN_DIR and REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)
original_python = repo / "environments/original/.venv/bin/python"
data_dir = Path("/kaggle/working/gqa-testdev-full-v1-s590")
data_manifest = (run_dir / "original-gqa-full-data/manifest.json") if RESUME_RUN_DIR else data_dir / "manifest.json"
report_path = run_dir / "original-gqa-full.json"
checkpoint_args = []  # Optional: ["--model", original_weight_path, "--vision-model", clip_path]
print("Run directory:", run_dir)


## 2. Reuse runtime and prepare full data

No new packages. Only the same two pinned GQA parquet files (~68 MB) are needed. The full run retains all question IDs without the pilot's group sampling. Native/project source snapshots are copied by the evaluator. In a new session, weights will download again unless attached.


In [ ]:
subprocess.run([
    "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
    "--output", str(run_dir / "original-runtime.json"),
], cwd=repo, check=True)
subprocess.run([str(original_python), "-I", "research/tests/test_gqa_full.py"], cwd=repo, check=True)
if not data_manifest.is_file():
    if RESUME_RUN_DIR:
        raise FileNotFoundError(f"Restored run is missing its data manifest: {data_manifest}")
    subprocess.run([
        str(original_python), "-I", "scripts/prepare_gqa_full.py",
        "--output-dir", str(data_dir), "--seed", "590", "--no-zip",
    ], cwd=repo, check=True)
data = json.loads(data_manifest.read_text())
print("Images:", len(data["images"]), "Questions:", len(data["questions"]))
print("Semantic types:", data["selection"]["type_counts"]["semantic"])
print("Expected scored predictions:", len(data["questions"]) * 4)


## 3. Run or resume

One model load across both T4s. Warmups are unscored. Raw JSONL flushes after every prediction; summary/ETA refresh every 100 questions. Rerun this cell to resume the same report. Resume checks protocol/data/code/lock/scoring identities and skips saved predictions. Interruption sends SIGINT to the child so it can save partial evidence; download Cell 4's ZIP before stopping the session.


In [ ]:
import signal

resume_args = ["--resume"] if report_path.exists() else []
command = [
    str(original_python), "-I", "scripts/eval_original_gqa_full.py",
    "--data-manifest", str(data_manifest),
    "--configurations", "baseline", "input-only", "decoder-only", "combined",
    "--max-new-tokens", "16", "--download-budget-gib", "20", "--report-every", "100",
    *checkpoint_args, *resume_args, "--output", str(report_path),
]
process = subprocess.Popen(command, cwd=repo)
try:
    returncode = process.wait()
except KeyboardInterrupt:
    if process.poll() is None:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    print("Stopped. Run Cell 4 to download partial evidence before ending the session.")
    raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


## 4. Inspect and download, including interrupted runs

Includes raw predictions, full question metadata/images, native/project source snapshots and runtime/protocol records. Excludes packages and model weights. A partial summary can lag JSONL after forced termination; resume rebuilds it. Keep this ZIP alongside the diagnostic pilot ZIP.


In [ ]:
import zipfile
from IPython.display import FileLink, display

if "process" in globals() and process.poll() is None:
    raise RuntimeError("The evaluator is still running. Interrupt Cell 3 and wait for it to stop before bundling.")
if report_path.is_file():
    result = json.loads(report_path.read_text())
    print("Status:", result["status"], "Progress:", result.get("progress"))
    if result.get("error"):
        print("Error:", result["error"])
    for name, metrics in result.get("summaries", {}).items():
        print(name, metrics["correct"], "/", metrics["evaluated"], "accuracy:", metrics["accuracy"],
              "APR:", result.get("comparisons", {}).get(name, {}).get("apr"))
bundle = run_dir.with_suffix(".zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.rglob("*")):
        if path.is_file() and path.suffix != ".lock":
            archive.write(path, str(path.relative_to(run_dir)))
print("Download before stopping the session:", bundle)
display(FileLink(str(bundle)))
